In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Setting up data

In [2]:
# to get same random numbers each time
np.random.seed(42)

# 90 days of daily check
n = 90

# list of 90 daily dates starting from today
dates = pd.date_range(end=pd.Timestamp.today(), periods=n, freq='D')

## 2. Building dataframe

In [3]:
df = pd.DataFrame({

    # dates for 90 days
    'date': dates,

    # random whole number 1-10 for each day
    'mood_rating': np.random.randint(1, 11, size=n),

    # sleep hours between 4-10 for each day
    'sleep_hours': np.random.choice( 
    np.arange(4, 9.5, 0.5), n),

    # stress level where 1 is low and 10 is high
    'stress_level': np.random.randint(1, 11, size=n),

    # 1 to 5 representing low to high social interaction
    'social_interaction': np.random.randint(1, 6, size=n),

    # exersise done 0 and 1 for each day and probability 40% for 0 and 60% for 1
    'exercise_done': np.random.choice([0,1], n, p=[0.4, 0.6]),

    # primary trigger that picks one of four moods trigger per day
    # probability of each trigger should add to 1.0

    'primary_trigger': np.random.choice(
    ['work', 'sleep', 'social', 'health', 'finance', 'relationship'],
    n,
    p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10])
})

In [5]:
# strip the time portion so only the date shows
df['date'] = df['date'].dt.normalize()

In [6]:
df.head()

,date,mood_rating,sleep_hours,stress_level,social_interaction,exercise_done,primary_trigger
0,2025-11-30,7,5.0,1,2,1,sleep
1,2025-12-01,4,4.0,5,3,1,social
2,2025-12-02,8,9.0,1,5,0,health
3,2025-12-03,5,6.0,8,1,0,sleep
4,2025-12-04,7,8.5,1,4,0,relationship


## 3. Mood score logic

In [7]:
# create a function to calculate mood_score
def calculate_mood_score(row):

    # start with 0 then adds points per day
    score = 0

    # mood_rating score is 1-10 muliplied by 4, contributes to max of 40 points
    score += row['mood_rating'] * 4

    # sleep_hours score is at total of 24 points, with 8 hours being optimal
    score += min(row['sleep_hours'], 8) * 3

    # stress_level score is 10 minus the stress level, muliplied by 2, contributes to max of 20 points
    score += (10 - row['stress_level']) * 2

    # social_interaction score is 1-5 muliplied by 2, contributes to max of 10 points
    score += row['social_interaction'] * 2

    # exercise_done score is 5 points if exercise done, 0 if not
    score += row['exercise_done'] * 5

    # caps the score at 100
    return min(round(score), 100)

# apply the function to each row and create a new column mood_score
df['mood_score'] = df.apply(calculate_mood_score, axis=1)
  

## 4. Recommendation logic

In [8]:
def get_recommendation(row):
   
    # if/elif to check the most urgent issue first

    # sleep deprivation is prioritised first as it affects everything else
    if row['sleep_hours'] < 6:
        return "Try to get at least 7 hours of sleep tonight"
        
     # high stress flagged second
    elif row['stress_level'] >= 7:
        return "Consider a short breathing exercise or go for a walk"
       
    # low social contact flagged third
    elif row['social_interaction'] <= 2:
        return "Reach out to someone you trust today"
    
    # no exercise flagged last
    elif row['exercise_done'] == 0:
        return "Even 10 minutes of movement can lift your mood"
        
   
    else:
        return "You're doing amazing — keep it up!"
    
# adds a new column with relevant recommendation for each day
df['recommendation'] = df.apply(get_recommendation, axis=1)


## 5. Saving data

In [9]:
#saving data to csv
df.to_csv('mindful_analytics_data.csv', index=False)

In [10]:
# first 5 row check
df.head()

,date,mood_rating,sleep_hours,stress_level,social_interaction,exercise_done,primary_trigger,mood_score,recommendation
0,2025-11-30,7,5.0,1,2,1,sleep,70,Try to get at least 7 hours of sleep tonight
1,2025-12-01,4,4.0,5,3,1,social,49,Try to get at least 7 hours of sleep tonight
2,2025-12-02,8,9.0,1,5,0,health,84,Even 10 minutes of movement can lift your mood
3,2025-12-03,5,6.0,8,1,0,sleep,44,Consider a short breathing exercise or go for ...
4,2025-12-04,7,8.5,1,4,0,relationship,78,Even 10 minutes of movement can lift your mood


In [11]:
# confirms scores are within a realistic range
print(f"Mood score range: {df['mood_score'].min()} - {df['mood_score'].max()}")

Mood score range: 29 - 86


## 6. Visualisation

### 6.1 Mood score over time

In [18]:
# area chart
fig1 = px.area(
    df,
    x='date',
    y='mood_score',
    title='Mood Score Over 90 Days',
    labels={'mood_score': 'Mood Score', 'date': 'Date'},
    color_discrete_sequence=["#86e5b3"]
)

fig1.add_hline(
    y=50,
    line_dash='dash',
    line_color='red',
    annotation_text='Midpoint (50)',
    annotation_position='bottom right'
)

fig1.update_layout(plot_bgcolor='white', yaxis_range=[0, 100])
fig1.show()


### 6.2 Primary Trigger chart

In [28]:
## counts how many times each trigger appears across the 90 days

# rename columns for clarity
trigger_counts = df['primary_trigger'].value_counts().reset_index()
trigger_counts.columns = ['trigger', 'count']

fig2 = px.pie(
    trigger_counts,
    names='trigger',        # the 6 trigger categories as the slice labels
    values='count',         # how often each trigger appeared
    title='What triggers mood the most?',
    hole=0.4,               # the hole size — 0.4 gives a clean donut shape
    color_discrete_sequence=px.colors.qualitative.Pastel
)
# adds the percentage and label inside each slice for better readability
fig2.update_traces(textposition='inside', textinfo='percent+label')

fig2.show()